<a href="https://colab.research.google.com/github/peremartra/fairness-pruning/blob/main/notebooks/03_neuron_bias_detection_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Neuron Bias Detection (English)

**Model:** `meta-llama/Llama-3.2-1B`  
**Dataset:** `oopere/fairness-pruning-pairs-en`  
**Categories:** `PhysicalAppearance` (15 pairs), `Gender` (15 pairs)  
**Library:** OptiPFair — `analyze_neuron_bias` + `compute_fairness_pruning_scores`

### Workflow
1. For each category: compute per-neuron bias scores from prompt pairs
2. Combine bias + structural importance → fairness pruning scores
3. Cross-category analysis: identify shared top candidates

### Outputs
```
fairness-pruning/results/llama-3.2-1B/bias_detection/en/
├── PhysicalAppearance/
│   ├── bias_scores.pt / bias_scores.json
│   ├── fairness_scores.pt / fairness_scores.json
├── Gender/
│   ├── bias_scores.pt / bias_scores.json
│   ├── fairness_scores.pt / fairness_scores.json
└── comparison_summary.json
```

## Cell 1 — Install dependencies

In [1]:
!pip install -q "optipfair[viz]" datasets transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.1/65.1 kB 7.1 MB/s eta 0:00:00


In [2]:
import random
import numpy as np
import torch

## Cell 2 — Configuration

Edit these values to change the experiment parameters.

In [3]:
# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME     = "meta-llama/Llama-3.2-3B"

# ── Dataset ────────────────────────────────────────────────────────────────
DATASET_NAME   = "oopere/fairness-pruning-pairs-en"
CATEGORIES     = ["PhysicalAppearance", "Gender", "Age",
                  "RaceEthnicity", "Religion"]

# ── OptiPFair — analyze_neuron_bias ────────────────────────────────────────
TARGET_LAYERS  = ["gate_proj", "up_proj"]   # GLU MLP projections to analyze
AGGREGATION    = "mean"                      # "mean" | "max" across sequence
BATCH_SIZE     = 4                           # Prompt pairs per batch (reduce if OOM)

# ── OptiPFair — compute_fairness_pruning_scores ────────────────────────────
# 0.0 = importance-only (standard pruning)  |  1.0 = bias-only
# 0.45 = balanced, slight edge toward structural importance
BIAS_WEIGHT    = 0.45

# ── Comparative analysis ───────────────────────────────────────────────────
# Percentile threshold to select top neuron candidates per layer
TOP_PERCENT    = 5.0

# ── Google Drive paths ─────────────────────────────────────────────────────
DRIVE_BASE     = "/content/drive/MyDrive/fairness-pruning"
DATASET_SLUG = DATASET_NAME.split("/")[-1].lower()   # → "fairness-pruning-pairs-en"
MODEL_SLUG   = MODEL_NAME.split("/")[-1].lower()   # → "llama-3.2-1b"
RESULTS_BASE = f"{DRIVE_BASE}/results/{MODEL_SLUG}/{DATASET_SLUG}"

In [4]:
def set_seed(seed=42):
    """Set random seed for reproducibility"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)

## Cell 3 — GPU detection and dtype assignment

In [5]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    compute_capability = torch.cuda.get_device_capability(0)

    # bfloat16 is natively supported from compute capability 8.0 onwards
    # (Ampere, Ada Lovelace, Hopper...). Below 8.0 (e.g. T4 = 7.5) use float16.
    if compute_capability[0] >= 8:
        DTYPE = torch.bfloat16
    else:
        DTYPE = torch.float16

    print(f"✅ GPU detected: {gpu_name}")
    print(f"   Compute capability: {compute_capability[0]}.{compute_capability[1]}")
    print(f"   VRAM: {vram_gb:.1f} GB")
else:
    DEVICE = "cpu"
    DTYPE = torch.float32
    print("⚠️  No GPU detected — running on CPU with float32 (slow)")

print(f"   dtype → {DTYPE}")

✅ GPU detected: NVIDIA L4
   Compute capability: 8.9
   VRAM: 23.7 GB
   dtype → torch.bfloat16


## Cell 4 — Mount Google Drive and create output directories

In [6]:
import os
from google.colab import drive

drive.mount("/content/drive")

# Create output directories for each category + the shared en/ root
dirs_to_create = [RESULTS_BASE] + [f"{RESULTS_BASE}/{cat}" for cat in CATEGORIES]

for d in dirs_to_create:
    os.makedirs(d, exist_ok=True)
    print(f"📁 {d}")

print("\n✅ Output directories ready.")

Mounted at /content/drive
📁 /content/drive/MyDrive/fairness-pruning/results/llama-3.2-3b/fairness-pruning-pairs-en
📁 /content/drive/MyDrive/fairness-pruning/results/llama-3.2-3b/fairness-pruning-pairs-en/PhysicalAppearance
📁 /content/drive/MyDrive/fairness-pruning/results/llama-3.2-3b/fairness-pruning-pairs-en/Gender
📁 /content/drive/MyDrive/fairness-pruning/results/llama-3.2-3b/fairness-pruning-pairs-en/Age
📁 /content/drive/MyDrive/fairness-pruning/results/llama-3.2-3b/fairness-pruning-pairs-en/RaceEthnicity
📁 /content/drive/MyDrive/fairness-pruning/results/llama-3.2-3b/fairness-pruning-pairs-en/Religion

✅ Output directories ready.


## Cell 5 — Load model and tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model: {MODEL_NAME} (dtype={DTYPE})")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
)
model.eval()

print(f"\n✅ Model loaded.")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"   Layers    : {model.config.num_hidden_layers}")

## Helper — serialization utilities

Both `bias_scores` (`Dict[str, Tensor]`) and `fairness_scores` (`Dict[int, Tensor]`)
are saved in two formats:
- `.pt` — native PyTorch, for downstream OptiPFair calls
- `.json` — human-readable, with `shape` stored separately from `values`

In [8]:
import json


def save_bias_scores(bias_scores: dict, out_dir: str, category):
    """Save bias_scores (Dict[str, Tensor]) as .pt and .json."""
    pt_path   = os.path.join(out_dir, f"{category}_bias_scores.pt")
    json_path = os.path.join(out_dir, f"{category}_bias_scores.json")

    # .pt — full tensors for OptiPFair
    torch.save(bias_scores, pt_path)

    # .json — shape + values for inspection
    serialized = {
        layer_name: {
            "shape" : list(tensor.shape),
            "values": tensor.cpu().float().tolist(),
        }
        for layer_name, tensor in bias_scores.items()
    }
    with open(json_path, "w") as f:
        json.dump(serialized, f, indent=2)

    print(f"   💾 bias_scores → {pt_path}")
    print(f"   💾 bias_scores → {json_path}")


def save_fairness_scores(fairness_scores: dict, out_dir: str, category):
    """Save fairness_scores (Dict[int, Tensor]) as .pt and .json."""
    pt_path   = os.path.join(out_dir, f"{category}_fairness_scores.pt")
    json_path = os.path.join(out_dir, f"{category}_fairness_scores.json")

    # .pt — full tensors
    torch.save(fairness_scores, pt_path)

    # .json — string keys (JSON requires string keys), shape + values
    serialized = {
        str(layer_idx): {
            "shape" : list(tensor.shape),
            "values": tensor.cpu().float().tolist(),
        }
        for layer_idx, tensor in fairness_scores.items()
    }
    with open(json_path, "w") as f:
        json.dump(serialized, f, indent=2)

    print(f"   💾 fairness_scores → {pt_path}")
    print(f"   💾 fairness_scores → {json_path}")


def load_prompt_pairs(dataset_name: str, category: str) -> list:
    """Load (prompt_1, prompt_2) tuples from a HuggingFace dataset subset."""
    from datasets import load_dataset
    ds = load_dataset(dataset_name, category, split="test")
    pairs = [(row["prompt_1"], row["prompt_2"]) for row in ds]
    print(f"   📂 {category}: {len(pairs)} prompt pairs loaded")
    return pairs


print("✅ Serialization utilities defined.")

✅ Serialization utilities defined.


# CELL 6.

In [ ]:
from optipfair.bias import analyze_neuron_bias, compute_fairness_pruning_scores

all_bias_scores     = {}
all_fairness_scores = {}

for category in CATEGORIES:
    out_dir = RESULTS_BASE
    #os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"  {category}")
    print(f"{'='*60}")

    # Load prompt pairs
    pairs = load_prompt_pairs(DATASET_NAME, category)

    # Bias analysis
    print(f"\n→ analyze_neuron_bias (aggregation='{AGGREGATION}', batch_size={BATCH_SIZE}) ...")
    bias_scores = analyze_neuron_bias(
        model=model,
        tokenizer=tokenizer,
        prompt_pairs=pairs,
        target_layers=TARGET_LAYERS,
        aggregation=AGGREGATION,
        show_progress=True,
    )
    save_bias_scores(bias_scores, out_dir, category)

    # Fairness pruning scores
    print(f"\n→ compute_fairness_pruning_scores (bias_weight={BIAS_WEIGHT}) ...")
    fairness_scores = compute_fairness_pruning_scores(
        model=model,
        bias_scores=bias_scores,
        bias_weight=BIAS_WEIGHT,
    )
    save_fairness_scores(fairness_scores, out_dir, category)

    all_bias_scores[category]     = bias_scores
    all_fairness_scores[category] = fairness_scores

    print(f"\n✅ {category} complete.")

print(f"\n{'='*60}")
print(f"All categories processed: {CATEGORIES}")

In [10]:
import math

# ── Configurable threshold ───────────────────────────────────────────────────
EXPLORE_TOP_PERCENT = 1.0

# ── Load fairness scores from disk (re-runnable independently) ───────────────
fs = {
    cat: torch.load(f"{RESULTS_BASE}/{cat}_fairness_scores.pt", map_location="cpu")
    for cat in CATEGORIES
}

# ── Global top-N% per category ───────────────────────────────────────────────
def top_neurons_global(fairness_scores: dict, top_percent: float) -> tuple:
    all_entries = [
        (score.item(), int(layer_idx), neuron_idx)
        for layer_idx, scores in fairness_scores.items()
        for neuron_idx, score in enumerate(scores)
    ]
    total = len(all_entries)
    k = max(1, math.ceil(total * top_percent / 100.0))
    all_entries.sort(key=lambda x: x[0], reverse=True)
    result = {}
    for _, layer_idx, neuron_idx in all_entries[:k]:
        result.setdefault(layer_idx, set()).add(neuron_idx)
    return result, k, total

top = {}
for cat in CATEGORIES:
    top[cat], k, total = top_neurons_global(fs[cat], EXPLORE_TOP_PERCENT)

# ── N-way intersection (neurons shared across ALL categories) ────────────────
all_layers = sorted(set(l for cat in CATEGORIES for l in top[cat]))

intersection = {}
for layer_idx in all_layers:
    sets = [top[cat].get(layer_idx, set()) for cat in CATEGORIES]
    shared = sorted(set.intersection(*sets))
    if shared:
        intersection[layer_idx] = shared

total_shared = sum(len(v) for v in intersection.values())

# ── Report ───────────────────────────────────────────────────────────────────
col_width = 8
header_cats = "".join(f"{cat[:6]:>{col_width}}" for cat in CATEGORIES)
print(f"Top {EXPLORE_TOP_PERCENT}% threshold (global ranking) — {len(CATEGORIES)} categories")
print(f"  N-way intersection (shared across ALL): {total_shared} neurons")
print()
print(f"  {'Layer':<8}{header_cats}{'Shared':>{col_width}}")
print(f"  {'-' * (8 + col_width * len(CATEGORIES) + col_width)}")

for layer_idx in all_layers:
    counts = "".join(f"{len(top[cat].get(layer_idx, set())):>{col_width}}" for cat in CATEGORIES)
    n_shared = len(intersection.get(layer_idx, []))
    marker = " ◀" if n_shared > 0 else ""
    print(f"  {layer_idx:<8}{counts}{n_shared:>{col_width}}{marker}")

# ── Save comparison_summary.json ─────────────────────────────────────────────
summary = {
    "config": {
        "model"        : MODEL_NAME,
        "dataset"      : DATASET_NAME,
        "bias_weight"  : BIAS_WEIGHT,
        "top_percent"  : EXPLORE_TOP_PERCENT,
        "target_layers": TARGET_LAYERS,
        "aggregation"  : AGGREGATION,
        "categories"   : CATEGORIES,
    },
    **{
        cat: {
            "top_neurons_by_layer": {str(k): sorted(v) for k, v in top[cat].items()},
            "total_candidates"    : sum(len(v) for v in top[cat].values()),
        }
        for cat in CATEGORIES
    },
    "intersection": {
        "neurons_by_layer"   : {str(k): v for k, v in intersection.items()},
        "total"              : total_shared,
        "layers_with_overlap": len(intersection),
    },
}

summary_path = f"{RESULTS_BASE}/comparison_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n💾 comparison_summary.json → {summary_path}")
print(f"✅ Cross-category analysis complete.")

Top 1.0% threshold (global ranking) — 5 categories
  N-way intersection (shared across ALL): 2 neurons

  Layer     Physic  Gender     Age  RaceEt  Religi  Shared
  --------------------------------------------------------
  0            435    1731     109     230     145       0
  1            425     126     546     320     133       0
  2             34     240      30      60       2       1 ◀
  3             42       5     305     236      36       0
  4              5       3      99      79      43       0
  5             16      11      25     128      97       0
  6             38       5      27      17      56       0
  7             17      10      23      66      76       0
  8             16       8       5      25      21       0
  9             34       7      10      70       6       0
  10            44       3      45      16       6       0
  11            15       9      32      65     241       0
  12            17       5       9     136     108       0
  13     

In [11]:
# ── Configurable threshold for this exploration ─────────────────────────────
EXPLORE_TOP_PERCENT = 0.5   # ← edit freely, independent of TOP_PERCENT above

# ── Load fairness scores from disk ──────────────────────────────────────────
fs = {
    cat: torch.load(f"{RESULTS_BASE}/{cat}_fairness_scores.pt", map_location="cpu")
    for cat in CATEGORIES
}

# ── Global top-N%: pool all neurons across all layers, rank globally ─────────
def top_neurons_global(fairness_scores: dict, top_percent: float) -> tuple:
    all_entries = [
        (score.item(), int(layer_idx), neuron_idx)
        for layer_idx, scores in fairness_scores.items()
        for neuron_idx, score in enumerate(scores)
    ]
    total = len(all_entries)
    k = max(1, math.ceil(total * top_percent / 100.0))
    all_entries.sort(key=lambda x: x[0], reverse=True)
    result = {}
    for _, layer_idx, neuron_idx in all_entries[:k]:
        result.setdefault(layer_idx, set()).add(neuron_idx)
    return result, k, total

top = {}
for cat in CATEGORIES:
    top[cat], k, total = top_neurons_global(fs[cat], EXPLORE_TOP_PERCENT)

# ── N-way intersection (shared across ALL categories) ────────────────────────
all_layers   = sorted(set(l for cat in CATEGORIES for l in top[cat]))
intersection = {}
for layer_idx in all_layers:
    sets   = [top[cat].get(layer_idx, set()) for cat in CATEGORIES]
    shared = sorted(set.intersection(*sets))
    if shared:
        intersection[layer_idx] = shared



total_shared = sum(len(v) for v in intersection.values())

# ── Report ───────────────────────────────────────────────────────────────────
col_w = 8
header = "".join(f"{cat[:6]:>{col_w}}" for cat in CATEGORIES)
print(f"Top {EXPLORE_TOP_PERCENT}% threshold (global ranking) — {len(CATEGORIES)} categories")
print(f"  Total neurons in model : {total}")
for cat in CATEGORIES:
    print(f"  {cat:<22}: {sum(len(v) for v in top[cat].values())} candidates")
print(f"  {'N-way intersection':<22}: {total_shared}")
print()
print(f"  {'Layer':<8}{header}{'Shared':>{col_w}}")
print(f"  {'-' * (8 + col_w * len(CATEGORIES) + col_w)}")
for layer_idx in all_layers:
    counts = "".join(f"{len(top[cat].get(layer_idx, set())):>{col_w}}" for cat in CATEGORIES)
    n_shared = len(intersection.get(layer_idx, []))
    marker = " ◀" if n_shared > 0 else ""
    print(f"  {layer_idx:<8}{counts}{n_shared:>{col_w}}{marker}")

Top 0.5% threshold (global ranking) — 5 categories
  Total neurons in model : 229376
  PhysicalAppearance    : 1147 candidates
  Gender                : 1147 candidates
  Age                   : 1147 candidates
  RaceEthnicity         : 1147 candidates
  Religion              : 1147 candidates
  N-way intersection    : 1

  Layer     Physic  Gender     Age  RaceEt  Religi  Shared
  --------------------------------------------------------
  0            198     848      41     102      85       0
  1            208      40     271     149      67       0
  2             22     101      19      37       2       1 ◀
  3             18       4     167     124      24       0
  4              4       3      44      46      15       0
  5             12      10      16      77      49       0
  6             34       4      15       8      34       0
  7             15       7      14      39      38       0
  8             13       6       4      17       9       0
  9             24       